# 🌱 Smart Farm IoT - Demonstração Machine Learning
## 🎯 Projeto de Mestrado - UNIOESTE
### 👨‍💻 Autor: Gustavo Felipe Paluch Figueiredo

---

**Objetivo:** Demonstrar aplicações de Machine Learning para agricultura de precisão

**Algoritmos Implementados:**
- Regressão Linear (Predição de Produtividade)
- K-Means (Agrupamento de Zones Agrícolas)
- Detecção de Anomalias (Sensores IoT)

**Dataset:** Dados simulados de sensores IoT agrícolas

## 1. 📥 Configuração e Importação de Bibliotecas

In [ ]:
# Bibliotecas essenciais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# Configurações de visualização
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

print("✅ Bibliotecas importadas com sucesso!")

## 2. 📊 Geração de Dataset Simulado

**Simulação de dados reais de sensores IoT:**
- Temperatura (°C)
- Umidade do solo (%)
- Umidade relativa do ar (%)
- Luminosidade (lux)
- Produtividade (kg/hectare) - Variável alvo

In [ ]:
# Gerando dataset simulado
np.random.seed(42)  # Para reproducibilidade
n_amostras = 500

# Sensores IoT simulados
temperatura = np.random.normal(25, 5, n_amostras)
umidade_solo = np.random.normal(60, 15, n_amostras)
umidade_ar = np.random.normal(70, 10, n_amostras)
luminosidade = np.random.normal(8000, 2000, n_amostras)

# Produtividade (relação com sensores + ruído)
produtividade = (
    1000 + 
    15 * temperatura + 
    8 * umidade_solo + 
    5 * umidade_ar + 
    0.02 * luminosidade +
    np.random.normal(0, 50, n_amostras)
)

# Criando DataFrame
df = pd.DataFrame({
    'temperatura': temperatura,
    'umidade_solo': umidade_solo,
    'umidade_ar': umidade_ar,
    'luminosidade': luminosidade,
    'produtividade': produtividade
})

print("📊 Dataset simulado criado:")
print(f"• Amostras: {len(df)}")
print(f"• Variáveis: {list(df.columns)}")
df.head()

## 3. 🔍 Análise Exploratória dos Dados

In [ ]:
# Estatísticas descritivas
print("📈 Estatísticas Descritivas:")
df.describe()

In [ ]:
# Visualização das distribuições
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('📊 Distribuição dos Dados dos Sensores IoT', fontsize=16)

df['temperatura'].hist(ax=axes[0,0], bins=20)
axes[0,0].set_title('Temperatura (°C)')

df['umidade_solo'].hist(ax=axes[0,1], bins=20)
axes[0,1].set_title('Umidade Solo (%)')

df['umidade_ar'].hist(ax=axes[0,2], bins=20)
axes[0,2].set_title('Umidade Ar (%)')

df['luminosidade'].hist(ax=axes[1,0], bins=20)
axes[1,0].set_title('Luminosidade (lux)')

df['produtividade'].hist(ax=axes[1,1], bins=20)
axes[1,1].set_title('Produtividade (kg/ha)')

# Correlação entre variáveis
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', ax=axes[1,2])
axes[1,2].set_title('Matriz de Correlação')

plt.tight_layout()
plt.show()

## 4. 🤖 Modelo 1: Regressão Linear - Predição de Produtividade

**Objetivo:** Prever a produtividade agrícola baseada nos dados dos sensores

In [ ]:
# Preparando dados para regressão
X = df[['temperatura', 'umidade_solo', 'umidade_ar', 'luminosidade']]
y = df['produtividade']

# Divisão treino/teste simplificada
X_train, X_test = X[:400], X[400:]
y_train, y_test = y[:400], y[400:]

# Treinando modelo de regressão linear
modelo_regressao = LinearRegression()
modelo_regressao.fit(X_train, y_train)

# Predições
y_pred = modelo_regressao.predict(X_test)

# Métricas de avaliação
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("🎯 REGRESSÃO LINEAR - Resultados:")
print(f"• MAE (Mean Absolute Error): {mae:.2f} kg/ha")
print(f"• R² Score: {r2:.4f}")
print(f"• Coeficientes: {modelo_regressao.coef_}")
print(f"• Intercept: {modelo_regressao.intercept_:.2f}")

In [ ]:
# Visualização das predições
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Produtividade Real (kg/ha)')
plt.ylabel('Produtividade Predita (kg/ha)')
plt.title('🤖 Regressão Linear: Real vs Predito')
plt.grid(True, alpha=0.3)
plt.show()

# Importância das features
importancias = pd.DataFrame({
    'feature': X.columns,
    'coeficiente': modelo_regressao.coef_
}).sort_values('coeficiente', key=abs, ascending=False)

print("\n📊 Importância das Variáveis:")
print(importancias)

## 5. 🎯 Modelo 2: K-Means - Zoneamento Agrícola

**Objetivo:** Identificar clusters/zones com características similares na lavoura

In [ ]:
# Preparando dados para clustering
X_cluster = df[['temperatura', 'umidade_solo', 'luminosidade']]

# Normalizando os dados
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Aplicando K-Means
kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

# Adicionando clusters ao DataFrame
df['cluster'] = clusters

print("🎯 K-MEANS CLUSTERING - Resultados:")
print(f"• Clusters identificados: 3")
print(f"• Distribuição dos clusters:")
print(df['cluster'].value_counts().sort_index())

In [ ]:
# Visualização dos clusters
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
scatter = plt.scatter(df['temperatura'], df['umidade_solo'], c=df['cluster'], cmap='viridis', alpha=0.7)
plt.xlabel('Temperatura (°C)')
plt.ylabel('Umidade Solo (%)')
plt.title('🌡️ Clusters: Temperatura vs Umidade Solo')
plt.colorbar(scatter, label='Cluster')

plt.subplot(1, 2, 2)
scatter = plt.scatter(df['luminosidade'], df['produtividade'], c=df['cluster'], cmap='viridis', alpha=0.7)
plt.xlabel('Luminosidade (lux)')
plt.ylabel('Produtividade (kg/ha)')
plt.title('💡 Clusters: Luminosidade vs Produtividade')
plt.colorbar(scatter, label='Cluster')

plt.tight_layout()
plt.show()

# Análise dos clusters
print("\n📊 Características dos Clusters:")
cluster_stats = df.groupby('cluster').agg({
    'temperatura': ['mean', 'std'],
    'umidade_solo': ['mean', 'std'],
    'produtividade': ['mean', 'std'],
    'luminosidade': ['mean', 'std']
}).round(2)

cluster_stats

## 6. 🚨 Modelo 3: Detecção de Anomalias em Sensores

**Objetivo:** Identificar leituras anômalas nos sensores IoT

In [ ]:
# Adicionando algumas anomalias artificiais
df_anomalias = df.copy()

# Inserindo anomalias (5% dos dados)
n_anomalias = int(0.05 * len(df))
anomalia_indices = np.random.choice(df.index, n_anomalias, replace=False)

df_anomalias.loc[anomalia_indices, 'temperatura'] = np.random.normal(40, 2, n_anomalias)  # Temperaturas altas
df_anomalias.loc[anomalia_indices, 'umidade_solo'] = np.random.normal(10, 3, n_anomalias)  # Umidade baixa

# Detecção de anomalias com Isolation Forest
X_anomalia = df_anomalias[['temperatura', 'umidade_solo', 'umidade_ar', 'luminosidade']]

iso_forest = IsolationForest(contamination=0.05, random_state=42)
anomalias = iso_forest.fit_predict(X_anomalia)

# -1 indica anomalia, 1 indica normal
df_anomalias['anomalia'] = anomalias
df_anomalias['anomalia_detectada'] = df_anomalias['anomalia'] == -1

print("🚨 DETECÇÃO DE ANOMALIAS - Resultados:")
print(f"• Total de amostras: {len(df_anomalias)}")
print(f"• Anomalias detectadas: {df_anomalias['anomalia_detectada'].sum()}")
print(f"• Taxa de detecção: {df_anomalias['anomalia_detectada'].mean():.2%}")

In [ ]:
# Visualização das anomalias detectadas
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
normal = df_anomalias[df_anomalias['anomalia_detectada'] == False]
anomalia = df_anomalias[df_anomalias['anomalia_detectada'] == True]

plt.scatter(normal['temperatura'], normal['umidade_solo'], 
            c='blue', alpha=0.6, label='Normal', s=50)
plt.scatter(anomalia['temperatura'], anomalia['umidade_solo'], 
            c='red', alpha=0.8, label='Anomalia', s=80, marker='x')
plt.xlabel('Temperatura (°C)')
plt.ylabel('Umidade Solo (%)')
plt.title('🚨 Detecção de Anomalias em Sensores')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
sns.boxplot(data=df_anomalias[['temperatura', 'umidade_solo', 'luminosidade']])
plt.title('📦 Boxplot - Detecção de Outliers')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Exemplo de anomalias detectadas
print("\n🔍 Exemplo de Anomalias Detectadas:")
anomalias_detectadas = df_anomalias[df_anomalias['anomalia_detectada'] == True].head()
anomalias_detectadas[['temperatura', 'umidade_solo', 'umidade_ar', 'luminosidade']]

## 7. 📈 Conclusões e Aplicações Práticas

In [ ]:
print("""
🎯 RESUMO DOS RESULTADOS OBTIDOS:

1. 🤖 REGRESSÃO LINEAR (Predição de Produtividade):
   • Modelo capaz de prever produtividade com boa precisão
   • Temperatura e umidade do solo são as variáveis mais importantes
   • Aplicação: Otimização de insumos baseada em predições

2. 🎯 K-MEANS (Zoneamento Agrícola):
   • Identificação de 3 zonas distintas na lavoura
   • Cada zona com características específicas de solo e clima
   • Aplicação: Agricultura de precisão com manejo diferenciado

3. 🚨 DETECÇÃO DE ANOMALIAS (Qualidade de Dados):
   • Sistema capaz de identificar leituras anômalas de sensores
   • Prevenção de tomadas de decisão baseadas em dados incorretos
   • Aplicação: Manutenção preditiva de sensores IoT

---
📚 PRÓXIMOS PASSOS PARA PESQUISA:
• Integração com dados reais de sensores IoT
• Desenvolvimento de modelos de séries temporais
• Implementação de alertas em tempo real
• Publicação dos resultados em periódico científico
""")

---

## 📊 **Referências Técnicas**

- Scikit-learn: Machine Learning em Python
- Pandas: Manipulação e análise de dados
- Matplotlib/Seaborn: Visualização de dados
- IoT Agriculture: Aplicações práticas de ML no agronegócio

**Repositório:** [Smart Farm IoT System](https://github.com/GustavoFelipe85/smart-farm-iot-system)

*Documento técnico elaborado para fins acadêmicos - UNIOESTE 2024*